# training a Standard PyTorch RNN (with Internals Inspection)

In this notebook, we:
1. **Load data**: Set up the separate datasets and data loaders.
2. **Define a Standard PyTorch RNN**: Build the next-word prediction model.
3. **Inspect the Model Internals**: Peer inside `nn.RNN` to look at its weight matrices, bias vectors, and manually compute a time-step forward pass to see exactly how PyTorch updates the hidden state under the hood.
4. **Train and Evaluate**: Train with gradient clipping and monitor Cross-Entropy Loss & Perplexity (PPL).
5. **Evaluate on Test split**: Verify final metrics on the unseen test set.
6. **Generate text (Inference)**: Feed prompt prefixes to generate new sentences.

## 1. Imports and Random Seed Setup

In [1]:
import os
import json
import math
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Set seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Seeds initialized.")

Seeds initialized.


## 2. Dataset and Loader Preparation

We load the train, validation, and test datasets.

In [3]:
class RNNNextWordDataset(Dataset):
    def __init__(self, data_path, split="train", format_type="sliding_window_format"):
        assert split in ["train", "val", "test"]
        assert format_type in ["sliding_window_format", "sentence_format"]
        
        with open(data_path, "r", encoding="utf-8") as f:
            data = json.load(f)
            
        self.inputs = data[format_type][split]["inputs"]
        self.targets = data[format_type][split]["targets"]
        
    def __len__(self):
        return len(self.inputs)
        
    def __getitem__(self, idx):
        x = torch.tensor(self.inputs[idx], dtype=torch.long)
        y = torch.tensor(self.targets[idx], dtype=torch.long)
        return x, y

def get_dataloaders(data_dir="Preprocessed_Data", format_type="sliding_window_format", batch_size=64):
    data_path = os.path.join(data_dir, "preprocessed_data.json")
    
    train_loader = DataLoader(RNNNextWordDataset(data_path, split="train", format_type=format_type), batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(RNNNextWordDataset(data_path, split="val", format_type=format_type), batch_size=batch_size, shuffle=False, drop_last=False)
    test_loader = DataLoader(RNNNextWordDataset(data_path, split="test", format_type=format_type), batch_size=batch_size, shuffle=False, drop_last=False)
    
    return train_loader, val_loader, test_loader

In [4]:
FORMAT_TYPE = "sliding_window_format"
BATCH_SIZE = 64

train_loader, val_loader, test_loader = get_dataloaders(
    data_dir="Preprocessed_Data",
    format_type=FORMAT_TYPE,
    batch_size=BATCH_SIZE
)

# Load vocab
vocab_path = os.path.join("Preprocessed_Data", "vocab.json")
with open(vocab_path, "r", encoding="utf-8") as f:
    vocab = json.load(f)
word2idx = vocab["word2idx"]
idx2word = {int(k): v for k, v in vocab["idx2word"].items()}
vocab_size = len(word2idx)

print(f"Dataset loaded. Vocab size: {vocab_size}")

Dataset loaded. Vocab size: 1221


## 3. Standard PyTorch RNN Model Definition

In [5]:
class PyTorchVanillaRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.dropout = nn.Dropout(dropout) # Dropout layer
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None):
        embed = self.embedding(x)  # [batch_size, seq_len, embedding_dim]
        embed = self.dropout(embed)  # Apply dropout to embeddings
        
        # Forward pass through standard RNN
        out, next_hidden = self.rnn(embed, hidden)  # out: [batch_size, seq_len, hidden_dim]
        
        out = self.dropout(out)  # Apply dropout to RNN outputs before classifier
        logits = self.fc(out)  # [batch_size, seq_len, vocab_size]
        return logits, next_hidden

## 4. Inspection Step: Peer Inside the Model

Let's see what parameters exist inside our standard model and check how the hidden state is computed step-by-step mathematically.

In [6]:
# Let's initialize a model locally for inspection
inspect_model = PyTorchVanillaRNN(vocab_size=vocab_size, embedding_dim=8, hidden_dim=16, dropout=0.5)

print("=== Model Parameters ===")
for name, param in inspect_model.named_parameters():
    print(f"Parameter: {name:<25} | Shape: {str(list(param.shape)):<20} | Requires Grad: {param.requires_grad}")

print("\n--- PyTorch RNN Parameter Explanations ---")
print("1. rnn.weight_ih_l0: Input-to-hidden weight matrix [hidden_dim, embedding_dim]")
print("2. rnn.weight_hh_l0: Hidden-to-hidden recurrence weight matrix [hidden_dim, hidden_dim]")
print("3. rnn.bias_ih_l0:   Input-to-hidden bias vector [hidden_dim]")
print("4. rnn.bias_hh_l0:   Hidden-to-hidden bias vector [hidden_dim]")

print("\n--- Mathematical Verification Cell ---")
# Create dummy input with batch_size=1, seq_len=3
dummy_input = torch.randint(0, vocab_size, (1, 3))
print(f"Dummy Input Shape: {dummy_input.shape}")

# Run built-in model
inspect_model.eval() # Crucial: eval() disables dropout so mathematical checks are clean!
with torch.no_grad():
    embed_val = inspect_model.embedding(dummy_input) # [1, 3, 8]
    rnn_out, rnn_hidden_final = inspect_model.rnn(embed_val)
    
# Extract RNN weights/biases
w_ih = inspect_model.rnn.weight_ih_l0
w_hh = inspect_model.rnn.weight_hh_l0
b_ih = inspect_model.rnn.bias_ih_l0
b_hh = inspect_model.rnn.bias_hh_l0

# Perform manual forward pass step-by-step:
h_prev = torch.zeros(1, 16)
manual_hidden_states = []

for t in range(3):
    x_t = embed_val[0, t, :]
    val_ih = torch.matmul(x_t, w_ih.t()) + b_ih
    val_hh = torch.matmul(h_prev, w_hh.t()) + b_hh
    h_t = torch.tanh(val_ih + val_hh)
    manual_hidden_states.append(h_t.unsqueeze(1))
    h_prev = h_t

manual_out = torch.cat(manual_hidden_states, dim=1)

difference = torch.abs(rnn_out - manual_out).max().item()
print(f"\nMax discrepancy between manual recurrence and PyTorch RNN: {difference:.2e}")
print("Matches perfectly!")

=== Model Parameters ===
Parameter: embedding.weight          | Shape: [1221, 8]            | Requires Grad: True
Parameter: rnn.weight_ih_l0          | Shape: [16, 8]              | Requires Grad: True
Parameter: rnn.weight_hh_l0          | Shape: [16, 16]             | Requires Grad: True
Parameter: rnn.bias_ih_l0            | Shape: [16]                 | Requires Grad: True
Parameter: rnn.bias_hh_l0            | Shape: [16]                 | Requires Grad: True
Parameter: fc.weight                 | Shape: [1221, 16]           | Requires Grad: True
Parameter: fc.bias                   | Shape: [1221]               | Requires Grad: True

--- PyTorch RNN Parameter Explanations ---
1. rnn.weight_ih_l0: Input-to-hidden weight matrix [hidden_dim, embedding_dim]
2. rnn.weight_hh_l0: Hidden-to-hidden recurrence weight matrix [hidden_dim, hidden_dim]
3. rnn.bias_ih_l0:   Input-to-hidden bias vector [hidden_dim]
4. rnn.bias_hh_l0:   Hidden-to-hidden bias vector [hidden_dim]

--- Mathematica

## 5. Hyperparameters and Configuration

In [7]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
DROPOUT = 0.5
LEARNING_RATE = 0.001
EPOCHS = 10
GRAD_CLIP = 5.0

model = PyTorchVanillaRNN(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, dropout=DROPOUT).to(DEVICE)

In [8]:

# Ignore pad index loss during cross entropy calculations
pad_idx = word2idx.get("<pad>", 0)
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)
print("\nChi tiết các tham số:")
for name, param in model.named_parameters():
    print(f"  - {name:<20} shape: {str(list(param.shape)):<20} | parameters: {param.numel():,}")

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTổng số tham số trainable của RNN (PyTorch Built-in): {total_params:,}")

PyTorchVanillaRNN(
  (embedding): Embedding(1221, 64)
  (dropout): Dropout(p=0.5, inplace=False)
  (rnn): RNN(64, 128, batch_first=True)
  (fc): Linear(in_features=128, out_features=1221, bias=True)
)

Chi tiết các tham số:
  - embedding.weight     shape: [1221, 64]           | parameters: 78,144
  - rnn.weight_ih_l0     shape: [128, 64]            | parameters: 8,192
  - rnn.weight_hh_l0     shape: [128, 128]           | parameters: 16,384
  - rnn.bias_ih_l0       shape: [128]                | parameters: 128
  - rnn.bias_hh_l0       shape: [128]                | parameters: 128
  - fc.weight            shape: [1221, 128]          | parameters: 156,288
  - fc.bias              shape: [1221]               | parameters: 1,221

Tổng số tham số trainable của RNN (PyTorch Built-in): 260,485


## 6. Training & Evaluation Loops

In [9]:
def train_epoch(model, loader, criterion, optimizer, device, grad_clip):
    model.train()
    total_loss = 0
    
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        
        optimizer.zero_grad()
        logits, _ = model(x)
        
        # Reshape logits to shape [batch * seq_len, vocab_size]
        loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
        
        loss.backward()
        
        # Clip gradients to prevent exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        
        optimizer.step()
        total_loss += loss.item()
        
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _ = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            total_loss += loss.item()
            
    avg_loss = total_loss / len(loader)
    perplexity = math.exp(avg_loss) if avg_loss < 20 else float('inf')
    return avg_loss, perplexity

## 7. Model Training Loop

In [10]:
best_val_loss = float("inf")

print("Training process initiated...")
for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, DEVICE, GRAD_CLIP)
    val_loss, val_ppl = evaluate(model, val_loader, criterion, DEVICE)
    
    print(f"Epoch {epoch:02d}/{EPOCHS:02d} | "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val PPL: {val_ppl:.2f}")
          
    # Save model weights if Val loss decreases
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_rnn_model.pth")
        print("--> Saved new best model checkpoint!")

Training process initiated...
Epoch 01/10 | Train Loss: 4.0985 | Val Loss: 3.5174 | Val PPL: 33.70
--> Saved new best model checkpoint!
Epoch 02/10 | Train Loss: 3.6208 | Val Loss: 3.3899 | Val PPL: 29.66
--> Saved new best model checkpoint!
Epoch 03/10 | Train Loss: 3.4782 | Val Loss: 3.3427 | Val PPL: 28.30
--> Saved new best model checkpoint!
Epoch 04/10 | Train Loss: 3.3953 | Val Loss: 3.3211 | Val PPL: 27.69
--> Saved new best model checkpoint!
Epoch 05/10 | Train Loss: 3.3359 | Val Loss: 3.3096 | Val PPL: 27.37
--> Saved new best model checkpoint!
Epoch 06/10 | Train Loss: 3.2902 | Val Loss: 3.2976 | Val PPL: 27.05
--> Saved new best model checkpoint!
Epoch 07/10 | Train Loss: 3.2541 | Val Loss: 3.2919 | Val PPL: 26.89
--> Saved new best model checkpoint!
Epoch 08/10 | Train Loss: 3.2234 | Val Loss: 3.2815 | Val PPL: 26.62
--> Saved new best model checkpoint!
Epoch 09/10 | Train Loss: 3.1971 | Val Loss: 3.2839 | Val PPL: 26.68
Epoch 10/10 | Train Loss: 3.1747 | Val Loss: 3.2794 |

## 8. Final Test Split Evaluation

In [9]:
# Load best checkpoint and run evaluation on the test split
model.load_state_dict(torch.load("best_rnn_model.pth", map_location=DEVICE))
test_loss, test_ppl = evaluate(model, test_loader, criterion, DEVICE)

print(f"Final Test Performance:")
print(f"  - Loss:       {test_loss:.4f}")
print(f"  - Perplexity: {test_ppl:.2f}")

Final Test Performance:
  - Loss:       3.2993
  - Perplexity: 27.09


## 9. Next Word Prediction (Inference)

Let's test our trained network on prompt prefixes to generate text outputs.

In [10]:
def clean_and_tokenize_prompt(prompt_text):
    text = prompt_text.lower()
    tokens = re.findall(r"[a-z0-9]+(?:'[a-z]+)?|[^\n\w\s]", text)
    return [t for t in tokens if t.strip()]

def generate_text(model, prompt, word2idx, idx2word, max_gen_len=15, temperature=0.8, seq_len=15, device="cpu"):
    model.eval()
    tokens = clean_and_tokenize_prompt(prompt)
    unk_id = word2idx.get("<unk>", 1)
    
    input_ids = [word2idx.get(t, unk_id) for t in tokens]
    generated_tokens = list(tokens)
    
    with torch.no_grad():
        for _ in range(max_gen_len):
            x_input = input_ids[-seq_len:] if len(input_ids) > seq_len else input_ids
            x_tensor = torch.tensor([x_input], dtype=torch.long, device=device)
            
            logits, _ = model(x_tensor)
            
            # Extract logits of the last time step
            next_logits = logits[0, -1, :] / temperature
            probs = torch.softmax(next_logits, dim=-1)
            next_idx = torch.multinomial(probs, num_samples=1).item()
            
            if next_idx == word2idx.get("<eos>", 3):
                break
                
            word = idx2word.get(next_idx, "<unk>")
            generated_tokens.append(word)
            input_ids.append(next_idx)
            
    return " ".join(generated_tokens)

# Sample Prompts
prompts = ["i don't", "what are we", "sheldon is"]
for prompt in prompts:
    out_str = generate_text(model, prompt, word2idx, idx2word, max_gen_len=12, temperature=0.8, seq_len=15, device=DEVICE)
    print(f"\nPrompt: \"{prompt}\"")
    print(f"Result: \"{out_str}\"")


Prompt: "i don't"
Result: "i don't know what about the <unk> ,"

Prompt: "what are we"
Result: "what are we talking to have to <unk> from the team ."

Prompt: "sheldon is"
Result: "sheldon is <unk> ."
